# Astra — ephemeral Kaggle local-AI server
Kaggle = **disposable compute**. State = Supabase/Postgres + object storage. Models = Kaggle Dataset. MCP/model workers = recreatable processes.

**Before running:** Settings → Accelerator **GPU T4 x1** (T4 x2 works but Astra pins GPU 0 by default) · Internet **On** · Add-ons → **Secrets**: `DATABASE_URL` (Supabase *pooler* URL), `SUPABASE_URL`, `SUPABASE_SERVICE_KEY`, `ASTRA_API_TOKEN`, optional `GITHUB_TOKEN` · add your **models dataset** and (optionally) the **astra-code dataset** as inputs.

Phase gates: §1–6 diagnostics → §7 Phase-1 smoke test (one model) → §8 Phase-2 swap test → only then the agent layer (§9+) is enabled. Every cell is idempotent.

## 1. Astra configuration

In [ ]:
ASTRA_REPO_URL   = ""          # optional: https://github.com/<you>/astra.git (used if the code dataset is not attached)
SMOKE_MODEL      = "logic"     # registry key used for the Phase-1 test (smallest model first is wise)
SOURCE_BUILD     = False       # True: compile llama-cpp-python for sm_75 if no CUDA wheel exists (10-20 min)
BUILD_LLAMA_SERVER = False     # True: build llama.cpp llama-server (needed for models with backend: llama_server)
RUN_SWAP_TEST    = True
START_API        = False       # True: start the HTTP gateway (needs ASTRA_API_TOKEN + a tunnel to be reachable from outside Kaggle)
CLEAN_ON_FAILURE = True

## 2. Environment detection + code locate (idempotent)

In [ ]:
import os, sys, glob, subprocess, pathlib
ON_KAGGLE = os.path.isdir("/kaggle/working")
print("on Kaggle:", ON_KAGGLE, "| python", sys.version.split()[0])
code_dir = None
for p in glob.glob("/kaggle/input/*/astra/bootstrap.py") + glob.glob("/kaggle/input/*/*/astra/bootstrap.py") + glob.glob("./astra/bootstrap.py") + glob.glob("../astra/bootstrap.py"):
    code_dir = str(pathlib.Path(p).resolve().parents[1]); break
if code_dir is None and ASTRA_REPO_URL:
    dest = "/kaggle/working/astra_src" if ON_KAGGLE else "./astra_src"
    if not os.path.isdir(dest):
        subprocess.run(["git", "clone", "--depth", "1", ASTRA_REPO_URL, dest], check=True)
    code_dir = dest
assert code_dir, "Astra source not found: attach the astra-code dataset or set ASTRA_REPO_URL"
if code_dir not in sys.path: sys.path.insert(0, code_dir)
print("astra code:", code_dir)
req = pathlib.Path(code_dir) / "requirements.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)

## 3–4. Persistent storage config + model dataset detection

In [ ]:
from astra.main import Astra
from astra import bootstrap
astra = Astra()
print("work_dir  :", astra.cfg.work_dir)
print("models_dir:", astra.cfg.models_dir)
print("secrets present:", astra.secrets.available())     # booleans only - values are never printed
for k in astra.reg.keys():
    v = astra.reg.validate(k); print(f"  {k:<13} {v['status']:<9} {v.get('size_mb','')} MB  {astra.reg.spec(k)['file']}")

## 5–6. Runtime validation + hardware diagnostics (Phase 1a)

In [ ]:
if BUILD_LLAMA_SERVER:
    print("llama-server:", bootstrap.build_llama_server(astra.cfg, astra.log))
rep = astra.boot("diagnostics", install_runtime=True, source_build=SOURCE_BUILD)
import json; print(json.dumps({k: rep[k] for k in ("compute", "runtime", "cuda_baseline_vram_mb", "timings_s")}, indent=1, default=str))

## 7. Phase 1 gate — load ONE model, run a prompt, report metrics, unload, verify cleanup

In [ ]:
from astra.selftest import smoke
res = smoke(astra, SMOKE_MODEL)
print(json.dumps(res, indent=1, default=str))
PHASE1_OK = res["passed"]
print("\nPHASE 1:", "PASSED" if PHASE1_OK else "FAILED  (see error/cleanup; fix before continuing)")

## 8. Phase 2 gate — strict one-model policy across all available specialists

In [ ]:
PHASE2_OK = False
if PHASE1_OK and RUN_SWAP_TEST:
    from astra.selftest import swap_test
    avail = astra.report["models"]["available"] or list(astra.reg.keys())
    sw = swap_test(astra, avail, rounds=1); print(json.dumps(sw, indent=1, default=str)); PHASE2_OK = sw["passed"]
print("PHASE 2:", "PASSED" if PHASE2_OK else "SKIPPED/FAILED")

## (Optional) Benchmark — find stable `gpu_layers` per model
Each model is benchmarked alone; results go to `astra/benchmarks/results/`, the DB (`model_metrics`) and an overlay (`best.json`) used by the registry.

In [ ]:
RUN_BENCHMARK = False
if RUN_BENCHMARK and PHASE1_OK:
    from astra.benchmarks.benchmark import run_benchmark
    astra.boot("persistence"); run_benchmark(astra, sweep=True)

## 9–10. Persistence connect, state restore, MCP bootstrap, server start, health check (agent layer)

In [ ]:
assert PHASE1_OK and (PHASE2_OK or not RUN_SWAP_TEST), "Phase gates not passed - agent layer stays disabled"
rep = astra.boot("full", install_runtime=False)
print(bootstrap.format_report(rep))

In [ ]:
if START_API:
    astra.serve()      # http://0.0.0.0:%d  (Bearer ASTRA_API_TOKEN). Kaggle has no inbound ports: use a tunnel (e.g. cloudflared) to reach it.
    print("gateway up on port", astra.cfg["server"]["port"])

## 11. Interactive control
`astra.run("goal")` blocks, prints progress, asks you questions inline (WAITING_FOR_USER never times out), then starts the **60 s** post-task countdown.
Other calls: `astra.status()` · `astra.submit(goal)` / `astra.answer(tid, text)` · `astra.resume(tid)` · `astra.mcp.status()` · `astra.mem.compact_context()`.

In [ ]:
astra.enable_console()
if astra.interrupted:
    for t in astra.interrupted:
        print("An interrupted task was found:", t["task_id"], "|", t["goal"][:80], "| step:", t["current_step"], "| last model:", t["last_model"])
    print("Resume it?  ->  astra.resume('<task_id>'); astra.wait('<task_id>')")

In [ ]:
task = astra.run("Describe what the workspace contains and summarize it for me.")   # edit the goal
print(task.status, "| idle shutdown in", astra.idle.remaining(), "s unless you continue (submit/answer/resume cancels it)")

## 12. Graceful shutdown
Runs the 13-step sequence, verifies persistence, and **refuses** if the state is not safely persisted (or the DB is the non-durable SQLite fallback) unless `force=True`.
It stops MCP + model workers and frees the GPU. It does **not** terminate the Kaggle VM (no supported API) — press *Stop session* in the UI, or use Save & Run All so the session ends when the notebook finishes.

In [ ]:
print(json.dumps(astra.shutdown("notebook_manual"), indent=1, default=str))